# Mat & Comp – ML 11: Reducción de dimensionalidad con PCA
## La mejor sombra

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 10

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. **Proyectar** datos sobre una dirección y medir cuánta **variación** conserva esa «sombra».
2. Encontrar a mano, girando un ángulo, la dirección que conserva más variación: la **primera componente principal**.
3. Calcular las componentes principales con numpy y con `PCA` de scikit-learn, y leer la **varianza explicada**.
4. Dibujar los 1797 dígitos de 64 píxeles en un **mapa de 2 dimensiones** y **comprimir** imágenes.
5. Usar PCA antes de otro modelo, y saber por qué hay que **normalizar** primero.

## 🔍 ¿Desde dónde tomas la foto?

Imagina que tienes que reconocer un objeto **solo por su sombra**. Una taza vista desde arriba da una sombra redonda: podría ser un plato, una moneda, un vaso… Vista de lado, se ven el asa y la forma: ¡es una taza!

✏️ *Si tuvieras que describir un carro con una sola foto en dos dimensiones, ¿desde qué ángulo la tomarías? ¿Por qué esa foto «conserva» más información que otras?*

**Mi respuesta:**

En ML 4 y ML 10 trabajamos con los dígitos, que tienen **64** características (una por píxel). No podemos dibujar 64 dimensiones. Hoy aprenderemos a encontrar la **mejor sombra** de los datos: unas pocas direcciones que conservan casi toda la información. Este método se llama **análisis de componentes principales** (PCA, por sus siglas en inglés).

## 1. De 2 dimensiones a 1: la sombra de un punto

Empecemos pequeño. Medimos la **estatura** y la **envergadura** (la distancia entre las puntas de los dedos con los brazos abiertos) de 100 estudiantes. Estos datos son simulados, pero en la vida real las dos medidas son casi iguales.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
estatura = rng.normal(162, 9, 100)
envergadura = estatura + rng.normal(0, 4, 100)
D = np.column_stack([estatura, envergadura])

plt.figure(figsize=(4.5, 4.5))
plt.scatter(D[:, 0], D[:, 1], s=15)
plt.xlabel("estatura (cm)"); plt.ylabel("envergadura (cm)")
plt.axis("equal"); plt.grid(alpha=0.3)
plt.title("100 estudiantes")
plt.show()

Los puntos forman una nube alargada en diagonal. Primero la **centramos**: le restamos el promedio a cada columna, para que el centro de la nube quede en $(0, 0)$.

Una **dirección** es un vector de largo 1, por ejemplo $u = (\cos\theta, \sin\theta)$. La **sombra** (proyección) de un punto $x$ sobre esa dirección es el número

$$x \cdot u = x_1 \cos\theta + x_2 \sin\theta$$

que dice qué tan lejos queda el punto a lo largo de esa dirección. Con 100 puntos obtenemos 100 sombras: hemos pasado de 2 números por estudiante a **1**.

In [ ]:
Dc = D - D.mean(axis=0)            # centrar

def sombras(datos, theta):
    u = np.array([np.cos(theta), np.sin(theta)])
    return datos @ u               # un número por punto

fig, ejes = plt.subplots(1, 3, figsize=(14, 4.2))
for eje, grados in zip(ejes, [0, 45, 135]):
    t = np.radians(grados)
    u = np.array([np.cos(t), np.sin(t)])
    s = sombras(Dc, t)
    proyectados = np.outer(s, u)   # dónde cae cada sombra sobre la línea
    eje.scatter(Dc[:, 0], Dc[:, 1], s=10, alpha=0.5)
    eje.plot([-35 * u[0], 35 * u[0]], [-35 * u[1], 35 * u[1]], color="crimson")
    eje.scatter(proyectados[:, 0], proyectados[:, 1], s=8, color="crimson")
    for p, q in zip(Dc[::5], proyectados[::5]):
        eje.plot([p[0], q[0]], [p[1], q[1]], color="gray", lw=0.5)
    eje.set_title(f"θ = {grados}°: varianza de las sombras = {s.var():.0f}")
    eje.set_xlim(-35, 35); eje.set_ylim(-35, 35); eje.set_aspect("equal")
plt.show()

La **varianza** de las sombras (ML 1: qué tan dispersos están) mide cuánta información conserva cada dirección:
- con θ = 45°, las sombras quedan muy separadas: se distinguen bien los estudiantes;
- con θ = 135°, las sombras quedan amontonadas en el centro: casi todos parecen iguales.

### Girar hasta encontrar la mejor dirección

Probemos todos los ángulos de 0° a 180°:

In [ ]:
angulos = np.arange(0, 180, 1)
varianzas = [sombras(Dc, np.radians(a)).var() for a in angulos]
mejor = angulos[np.argmax(varianzas)]

plt.figure(figsize=(6.5, 3))
plt.plot(angulos, varianzas)
plt.axvline(mejor, color="crimson", ls="--", label=f"mejor ángulo ≈ {mejor}°")
plt.xlabel("ángulo θ (grados)"); plt.ylabel("varianza de las sombras")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f"Varianza total de los datos (suma de las dos columnas): {Dc.var(axis=0).sum():.1f}")
print(f"Varianza en la mejor dirección ({mejor}°):              {max(varianzas):.1f}")
print(f"Porcentaje conservado: {max(varianzas) / Dc.var(axis=0).sum():.1%}")

La mejor dirección conserva casi toda la variación con **un solo número** por estudiante. Ese número es algo como «tamaño del estudiante». La dirección perpendicular (la segunda) solo guarda las pequeñas diferencias entre estatura y envergadura.

Esta mejor dirección es la **primera componente principal**.

## 2. PCA con numpy: sin girar ángulos

Girar ángulos funciona en 2 dimensiones, pero con 64 sería imposible probar todas las direcciones. Con álgebra lineal se demuestra que las mejores direcciones son los **vectores propios** de la **matriz de covarianza** de los datos centrados, y que cada **valor propio** es justamente la varianza en esa dirección. numpy los calcula con `np.linalg.eigh`:

In [ ]:
cov = Dc.T @ Dc / len(Dc)                  # matriz de covarianza (2 × 2)
valores, vectores = np.linalg.eigh(cov)    # eigh los da de menor a mayor
orden = np.argsort(valores)[::-1]
valores, vectores = valores[orden], vectores[:, orden]

print("Matriz de covarianza:\n", cov.round(1))
print("\nVarianzas (valores propios):", valores.round(1))
pc1 = vectores[:, 0]
angulo = np.degrees(np.arctan2(pc1[1], pc1[0])) % 180
print(f"Primera componente: {pc1.round(3)}  →  ángulo {angulo:.1f}°")

¡El mismo ángulo que encontramos girando! Y la varianza de la primera componente es la misma que el máximo de la curva.

### Con scikit-learn

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(D)       # PCA centra los datos por dentro
print("Componentes (una por fila):\n", pca.components_.round(3))
print("Varianza explicada:", pca.explained_variance_.round(1))
print("Proporción de varianza explicada:", pca.explained_variance_ratio_.round(3))

(Puede que el signo de una componente salga al revés: $u$ y $-u$ son la misma línea. Y la varianza sale un poquito mayor, 163.1 en lugar de 161.5, porque scikit-learn divide por $n - 1$ en lugar de $n$ al calcularla.)

`explained_variance_ratio_` dice qué fracción de toda la variación conserva cada componente.

## 3. El mapa de los dígitos: de 64 dimensiones a 2

Ahora sí, los 1797 dígitos de 8 × 8 píxeles. Cada imagen es un punto en un espacio de **64 dimensiones**. Busquemos sus 2 mejores sombras:

In [ ]:
from sklearn.datasets import load_digits

dig = load_digits()
Xd, yd = dig.data, dig.target

pca2 = PCA(n_components=2)
Z = pca2.fit_transform(Xd)            # 1797 × 2
print("Antes:", Xd.shape, "  después:", Z.shape)
print(f"Varianza conservada con 2 componentes: {pca2.explained_variance_ratio_.sum():.1%}")

plt.figure(figsize=(8, 6.5))
puntos = plt.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=8, alpha=0.7)
for d in range(10):
    cx, cy = np.median(Z[yd == d], axis=0)
    plt.text(cx, cy, str(d), fontsize=18, weight="bold",
             bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
plt.xlabel("componente 1"); plt.ylabel("componente 2")
plt.title("Los 1797 dígitos vistos en 2 dimensiones")
plt.colorbar(puntos, ticks=range(10))
plt.show()

Estas 2 sombras conservan solo cerca del 28 % de la variación, y ¡PCA **no** conoce las etiquetas! Solo buscó las direcciones de mayor variación. Aun así, dígitos parecidos quedan cerca: los 0 abajo, los 4 a la derecha, los 3 cerca de los 9, y los 5 y 8 mezclados en el centro.

✏️ *¿Qué dígitos quedan bien separados? ¿Cuáles se mezclan? ¿Coincide con los errores de k vecinos en ML 4?*

### ¿Qué «miran» las componentes?

Cada componente es un vector de 64 números: ¡podemos dibujarlo como una imagen de 8 × 8!

In [ ]:
pca10 = PCA(n_components=10).fit(Xd)
fig, ejes = plt.subplots(1, 6, figsize=(13, 2.5))
ejes[0].imshow(pca10.mean_.reshape(8, 8), cmap="gray_r")
ejes[0].set_title("promedio")
for k in range(5):
    lim = np.abs(pca10.components_[k]).max()
    ejes[k + 1].imshow(pca10.components_[k].reshape(8, 8), cmap="RdBu", vmin=-lim, vmax=lim)
    ejes[k + 1].set_title(f"componente {k + 1}")
for e in ejes:
    e.axis("off")
plt.show()

Cada imagen se describe como **el promedio** más un poco de la componente 1, más un poco de la 2… Las sombras `Z` son justamente esos «pocos»: cuánto de cada componente lleva la imagen.

## 4. ¿Cuántas componentes? Comprimir y reconstruir

Si guardamos solo $k$ números por imagen, ¿cuánto se parece la imagen reconstruida a la original? `inverse_transform` hace el camino de vuelta: promedio + suma de componentes.

In [ ]:
pca_todo = PCA().fit(Xd)
acumulada = np.cumsum(pca_todo.explained_variance_ratio_)

plt.figure(figsize=(6.5, 3.2))
plt.plot(range(1, 65), acumulada, "o-", ms=3)
plt.axhline(0.9, color="gray", ls="--")
plt.xlabel("número de componentes"); plt.ylabel("varianza conservada")
plt.title("Varianza acumulada de los dígitos")
plt.grid(alpha=0.3)
plt.show()

for k in [2, 5, 10, 20, 40]:
    print(f"{k:2} componentes conservan el {acumulada[k - 1]:.1%} de la variación")

In [ ]:
ks = [2, 5, 10, 20, 64]
ejemplos = [0, 1, 2, 3]
fig, ejes = plt.subplots(len(ejemplos), len(ks), figsize=(9, 7.5))
for j, k in enumerate(ks):
    p = PCA(n_components=k).fit(Xd)
    reconstruidas = p.inverse_transform(p.transform(Xd[ejemplos]))
    for i in range(len(ejemplos)):
        ejes[i, j].imshow(reconstruidas[i].reshape(8, 8), cmap="gray_r")
        ejes[i, j].axis("off")
    ejes[0, j].set_title(f"{k} números")
plt.suptitle("Reconstruir con k componentes (64 = la original)")
plt.show()

Con **2** números apenas se adivina el dígito; con **10 a 20** (en lugar de 64) ya se reconocen bien. Así funcionan muchas formas de **comprimir** datos: guardar solo lo que más varía.

## 5. PCA antes de otro modelo

Menos características significa modelos más rápidos y, a veces, menos sobreajuste. Probemos k vecinos (ML 4) con los dígitos reducidos a distintas dimensiones, usando validación cruzada (ML 7) y un `Pipeline`:

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
for k in [2, 5, 10, 20, 64]:
    modelo = make_pipeline(PCA(n_components=k), KNeighborsClassifier(5))
    exactitud = cross_val_score(modelo, Xd, yd, cv=5).mean()
    print(f"{k:2} componentes: exactitud {exactitud:.1%}")

Con 2 componentes el mapa es bonito, pero se pierde demasiado para clasificar. Con 10 a 20 componentes, k vecinos funciona tan bien (o casi) como con los 64 píxeles.

💡 Fíjate que el `PCA` va **dentro** del pipeline: así, en cada grupo de la validación cruzada, PCA se ajusta solo con los datos de entrenamiento de ese grupo, sin espiar los de prueba (ML 7).

## 6. ⚠️ Normalizar primero

PCA busca la dirección de mayor **varianza**. Si una columna tiene números mucho más grandes que las otras, gana por tamaño y no por importancia. Volvamos a los estudiantes de ML 1:

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(URL)
datos["horas_sueno"] = datos["horas_sueno"].fillna(datos["horas_sueno"].mean())
columnas = ["horas_estudio", "horas_sueno", "horas_redes", "nota_matematicas"]
E = datos[columnas].to_numpy()
print("Varianza de cada columna:", dict(zip(columnas, E.var(axis=0).round(2))))

In [ ]:
from sklearn.preprocessing import StandardScaler

sin_normalizar = PCA(n_components=1).fit(E)
normalizado = make_pipeline(StandardScaler(), PCA(n_components=1)).fit(E)

fig, ejes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
for eje, comp, titulo in [(ejes[0], sin_normalizar.components_[0], "sin normalizar"),
                          (ejes[1], normalizado[-1].components_[0], "normalizado")]:
    eje.barh(columnas, comp, color=["seagreen" if c > 0 else "crimson" for c in comp])
    eje.axvline(0, color="k", lw=1)
    eje.set_title(f"Componente 1, {titulo}")
plt.tight_layout()
plt.show()

- **Sin normalizar**, la primera componente es casi solo `horas_estudio`, porque es la columna con números más grandes (y más varianza).
- **Normalizada**, la componente combina todo: más estudio, más sueño, mejor nota y menos redes van juntos. Podríamos llamarla «hábitos de buen estudiante». (El signo de toda la componente puede salir al revés; lo que importa es qué columnas van juntas.)

Igual que con k vecinos (ML 4) y con Ridge y Lasso (ML 9): **normaliza antes de PCA** cuando las columnas tienen unidades distintas.

## 7. Errores comunes

**Error 1: pedir más componentes que columnas.** Con 4 columnas hay como máximo 4 direcciones:

In [ ]:
PCA(n_components=10).fit(E)

**Error 2: no centrar** al hacer PCA a mano. Sin restar el promedio, la «mejor dirección» apunta hacia el centro de la nube en lugar de a lo largo de ella. `PCA` de scikit-learn centra solo.

**Error 3: no normalizar** columnas con unidades distintas (sección 6).

**Error 4: creer que PCA separa las clases.** PCA busca variación, no clases. Muchas veces coinciden (como en los dígitos), pero no siempre: la dirección que más varía puede no tener nada que ver con lo que queremos predecir.

**Error 5: ajustar PCA con los datos de prueba.** PCA también «aprende» (el promedio y las direcciones). Ajústalo solo con el entrenamiento; un `Pipeline` lo hace por ti.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en la sección 1 haces la envergadura **independiente** de la estatura (`envergadura = rng.normal(162, 9, 100)`)? ¿Cómo queda la curva de varianza contra ángulo? ¿Cuánto conserva la mejor dirección?
2. …calculas las sombras con los datos **sin centrar** (`D` en lugar de `Dc`) y buscas el ángulo de mayor `(sombras ** 2).mean()`? ¿A dónde apunta?
3. …sumas `pca.explained_variance_ratio_` para las dos componentes de la sección 2?
4. …haces el mapa de los dígitos con las componentes 3 y 4 en lugar de 1 y 2?
5. …reconstruyes un dígito con **1** sola componente? ¿Se parece a algo?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). Sombras a mano.** Calcula sin computador la sombra del punto $(3, 4)$ sobre la dirección $(1, 0)$, sobre $(0, 1)$ y sobre $(0.6, 0.8)$. ¿Por qué la última es la más grande? Compruébalo con numpy.

**Ejercicio 2 (fácil). ¿Cuántas para el 90 %?** Usando `acumulada` de la sección 4, encuentra el menor número de componentes que conserva al menos el 90 % de la variación de los dígitos. ¿Y el 95 %? ¿Y el 99 %? (Pista: `np.argmax(acumulada >= 0.9) + 1`.)

**Ejercicio 3 (medio). Error de reconstrucción.** Para $k$ = 1, 2, …, 40, reconstruye **todos** los dígitos con $k$ componentes y calcula el error cuadrático medio entre las imágenes originales y las reconstruidas. Grafica el error contra $k$. ¿Qué relación tiene con la curva de varianza acumulada?

**Ejercicio 4 (medio). El mapa por clases.** En el mapa de 2 dimensiones de los dígitos, calcula el **centro** (promedio de `Z`) de cada dígito. ¿Qué par de dígitos tiene los centros más cercanos? ¿Y los más lejanos? ¿Tiene sentido según su forma?

**Ejercicio 5 (avanzado). PCA de los dígitos a mano.** Centra `Xd`, calcula la matriz de covarianza (64 × 64) y sus vectores propios con `np.linalg.eigh`, ordénalos de mayor a menor valor propio y compara las dos primeras componentes con `pca2.components_` (pueden diferir en el signo). Calcula también la proporción de varianza de cada una y compárala con `explained_variance_ratio_`.

## 🏆 Reto opcional: comprimir una foto

scikit-learn trae una foto de ejemplo: `from sklearn.datasets import load_sample_image` y `foto = load_sample_image("china.jpg")`.

1. Conviértela a gris con `gris = foto.mean(axis=2)`. ¿Qué forma tiene?
2. Trata cada **fila** de la imagen como un dato (cada fila es un vector de 640 números) y ajusta `PCA` con 10, 30 y 100 componentes. Reconstruye y muestra las imágenes.
3. ¿Cuántos números hay que guardar en cada caso? (Pista: las sombras de cada fila, más las componentes, más el promedio.) ¿Qué porcentaje es del tamaño original?

## 🎬 Cierre

Hoy aprendiste que:
- La **sombra** (proyección) de un punto sobre una dirección $u$ es $x \cdot u$, y la **varianza** de las sombras mide cuánta información conserva esa dirección.
- La dirección de mayor varianza es la **primera componente principal**; se encuentra girando (en 2D) o con los **vectores propios** de la matriz de covarianza.
- La **varianza explicada** dice cuántas componentes hacen falta: con 20 de 64 se conserva casi el 90 % de la variación de los dígitos.
- PCA sirve para **visualizar** (el mapa de los dígitos), **comprimir** y **simplificar** antes de otro modelo.
- Hay que **normalizar** cuando las columnas tienen unidades distintas, y PCA no conoce las etiquetas.

**Para pensar:** ya tienes en tu caja de herramientas pandas, regresiones, k vecinos, árboles, bosques, k-means, PCA, validación cruzada… En la **última lección** de esta sección harás un **proyecto completo**: partir de datos crudos y desordenados, limpiarlos, explorarlos, elegir y comparar modelos con honestidad, y presentar un resultado evaluado. 🏁